# A nonlinear electric circuit

This notebook generates observations of an electric circuit, fits a model, and compares its predictions with the true system.

The state is charge $q$ and magnetic flux $\phi$, so that $x=(q,\phi)$. For capacitance $C$, current scale $I_s$, and flux scale $\delta$, the stored energy is
$$H(q,\phi)=\frac{q^2}{2C}+I_s\delta\log\cosh(\phi/\delta).$$
The current is $i=I_s\tanh(\phi/\delta)$. With resistance $r$, applied voltage $u(t)$ and forcing gain $g(q)$, the circuit is governed by the equations
$$\dot q=i,\qquad \dot\phi=-q/C-ri+g(q)u(t).$$

The circuit model is adapted from [Castaños and coauthors (2009)](https://doi.org/10.1007/s00034-009-9103-x). Note that a polynomial model can only approximate its nonlinear energy and forcing gain.

## Settings

Choose the dataset size, noise level, physical parameters and model components below. Increase `max_steps` and `refit_steps` for longer fitting, or set `noise_fraction` above zero to add measurement noise. Use even trajectory counts when learning the forcing gain.

These small demonstration settings are not the paper settings; short fits can give substantial prediction errors.

For analytic damping, `damping_library="polynomial"` uses a constant and squared flux; `"gaussian"` also adds Gaussian functions.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

import phsi

# Works when launched from either the repository root or examples/.
examples = (
    Path.cwd() if (Path.cwd() / "_systems.py").exists() else Path.cwd() / "examples"
)
sys.path.insert(0, str(examples.resolve()))
from _systems import generate_dataset, plot_observations, plot_predictions, save_dataset

settings = {
    "case": "electric_circuit",
    "variant": "constant_unknown_gain",
    "seed": 20261029,
    "n_train": 4,
    "n_val": 2,
    "n_test": 2,
    "duration": 5.0,
    "dt": 0.05,
    "noise_fraction": 0.0,
    "initial_ranges": [1.0, 1.5],
    "amplitude": 0.3,
    "omegas": [0.5, 0.9, 1.4, 2.1],
    "physics": {
        "capacitance": 1.0,
        "resistance": 0.15,
        "current_scale": 1.0,
        "flux_scale": 0.7,
        "gain_scale": 0.75,
    },
}

# None picks defaults for the selected physical variant.
energy_component = "analytic"  # "analytic" or "nn"
damping_component = None  # "constant", "analytic", or "nn"
gain_component = None  # "known", "analytic", or "nn"
energy_library = "polynomial"  # "polynomial" or "gaussian"
damping_library = "gaussian"  # "polynomial" or "gaussian"
gain_library = "polynomial"  # "polynomial" or "gaussian"
polynomial_degree = 4
fit_method = "integration_scheme"  # "integration_scheme", "simulation_error", "derivative", or "midpoint"
max_steps = 300
refit_steps = 50
learning_rate = 0.001
prune = True
preprocessing = None  # or "savgol": an 11-sample cubic Savitzky–Golay smoother
output = examples / "output" / settings["case"]

## Generate observations

Simulate the system to create training, validation and test data. Change the reference laws in `_systems.py` to try a different system.

In [ ]:
dataset = generate_dataset(settings)
settings["learner"] = dict(
    energy=energy_component,
    damping=damping_component,
    gain=gain_component,
    energy_library=energy_library,
    damping_library=damping_library,
    gain_library=gain_library,
    polynomial_degree=polynomial_degree,
    method=fit_method,
    max_steps=max_steps,
    refit_steps=refit_steps,
    learning_rate=learning_rate,
    prune=prune,
    preprocessing=preprocessing,
)
save_dataset(dataset, settings, output)
train, validation, test = (dataset[name] for name in ("train", "val", "test"))
print({name: dataset[name]["clean"].shape for name in ("train", "val", "test")})

state_names = ["q", "φ"]
observation_figure = plot_observations(dataset, state_names, output)
display(observation_figure)
plt.close(observation_figure)

## Fit and print the model

Build the model chosen in the settings and fit it to the training observations. Validation data guide model selection. The next cell prints the fitted equations.

Choose analytic functions, neural networks, or a mix of both in the settings. The `derivative` and `midpoint` fitting methods require analytic components.

In [ ]:
variant = settings["variant"]
damping_choice = damping_component or (
    "analytic" if variant == "state_dependent_known_gain" else "constant"
)
gain_choice = gain_component or (
    "known" if variant == "state_dependent_known_gain" else "analytic"
)


def analytic_library(kind):
    return (
        phsi.Polynomial(degree=polynomial_degree)
        if kind == "polynomial"
        else phsi.GaussianExtension(degree=polynomial_degree)
    )


hamiltonian = (
    analytic_library(energy_library)
    if energy_component == "analytic"
    else phsi.Neural(hidden=(64, 64))
)
damping = (
    phsi.DampingLibrary(kind=damping_library)
    if damping_choice == "analytic"
    else phsi.Neural(hidden=(16, 16))
    if damping_choice == "nn"
    else "constant"
)
gain = (
    analytic_library(gain_library)
    if gain_choice == "analytic"
    else phsi.Neural(hidden=(16, 16))
    if gain_choice == "nn"
    else "known"
)
model = phsi.model(
    hamiltonian=hamiltonian,
    damping=damping,
    forcing_gain=gain,
    gain_state=0,
    state_names=state_names,
)
fit_options = dict(
    method=fit_method, seed=settings["seed"], prune=prune, preprocessing=preprocessing
)
if fit_method in ("integration_scheme", "simulation_error"):
    fit_options.update(
        max_steps=max_steps, refit_steps=refit_steps, learning_rate=learning_rate
    )
model.fit(
    train["observed"],
    t=train["t"],
    u=train["signals"],
    validation=(validation["observed"], validation["t"], validation["signals"]),
    **fit_options,
)

In [ ]:
model.print(expand=True, decimals=4)

## Predict new trajectories

Compare the fitted model with the true system using new initial conditions and the same applied input.

In [ ]:
predictions = []
for x0, signal in zip(test["x0"], test["signals"]):
    try:
        x_pred = model.simulate(x0, t=test["t"], u=signal)
        predictions.append(dict(t=test["t"], x=x_pred, success=True, diagnostics={}))
    except phsi.SimulationError as error:
        predictions.append(
            dict(t=error.t, x=error.x, success=False, diagnostics=error.diagnostics)
        )

figures = plot_predictions(dataset, predictions, state_names, output)
for figure in figures:
    display(figure)
    plt.close(figure)

## Save the model

Save the fitted model and load it again. Rerunning this cell replaces the previous model file. The output folder also contains the data and figures.

See the [method notes](../docs/methods.md) and [paper recipes](../reproduce/README.md) for more detail.

In [ ]:
model.save(output / "model.phsi", overwrite=True)
restored = phsi.load(output / "model.phsi")
print("Saved demonstration artifacts to", output.resolve())